<a href="https://colab.research.google.com/github/stella-s-678/stella-s-678-DataScience-GenAI-Submissions/blob/main/Copy_of_7_02_Machine_Learning_Modelling_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Modelling – Loan Data

**Week 7 Seminar | Option B**

Build and compare classification models using `loan_data.csv`. All features are numeric, with no missing values. The label is `loan_status`. Complete the tasks below in Google Colab.

**Student exercise:** Write your own code in the empty cells.

## Task 1 – Load and Inspect the Dataset

Upload `loan_data.csv` to Colab and load it into a dataframe. Inspect the first few rows, dataset dimensions, column types, and the distribution of `loan_status`. All columns are numeric and there are no missing values; verify these properties.

In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Saving loan_data.csv to loan_data.csv
   age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amou

## Task 2 – Separate Predictors and Target

Set `loan_status` as the target (Y) and use the remaining columns as predictors (X). Check the dimensions of both.

In [ ]:

X = df.drop('loan_status', axis=1)
y = df['loan_status']

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (50000, 15)
y shape: (50000,)


## Task 3 – Split the Data

Create training and test sets, using 80% of the observations for training and 20% for testing. Preserve the class proportions and use a fixed random seed so your results are reproducible. Check the resulting shapes.

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (40000, 15)
X_test: (10000, 15)
y_train: (40000,)
y_test: (10000,)


## Task 4 – Scale the Predictors

Scale the predictors so they are on a comparable scale. Fit the scaling procedure using **training data only**, then apply the same transformation to the test data. Why should the test set not be used to fit the scaler?

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Training:', X_train_scaled.shape)
print('Test:', X_test_scaled.shape)

Training: (40000, 15)
Test: (10000, 15)


## Task 5 – Train Machine Learning Models

Train and compare **logistic regression**, **random forest**, and **gradient boosting decision trees (GBDT)** with reasonable initial settings. Use the scaled predictors for logistic regression. Tree-based models do not require scaling, so you can use the unscaled predictors for them.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

# 2. Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# 3. Gradient Boosting Decision Trees
gbdt = GradientBoostingClassifier(
    n_estimators=100,
    random_state=42
)
gbdt.fit(X_train, y_train)

print('All three models trained successfully!')

All three models trained successfully!


## Task 6 – Evaluate and Compare Models

Evaluate each model on **both the training and test sets**. Report accuracy, macro precision, macro recall, and macro F1-score. Compare training and test performance. Which model generalizes best, and is there evidence of overfitting?

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Logistic Regression': (lr, X_train_scaled, X_test_scaled),
    'Random Forest': (rf, X_train, X_test),
    'GBDT': (gbdt, X_train, X_test)
}

results = []

for name, (model, Xtr, Xte) in models.items():
    for dataset, Xdata, ydata in [
        ('Train', Xtr, y_train),
        ('Test', Xte, y_test)
    ]:
        y_pred = model.predict(Xdata)

        results.append({
            'Model': name,
            'Dataset': dataset,
            'Accuracy': accuracy_score(ydata, y_pred),
            'Macro Precision': precision_score(ydata, y_pred, average='macro', zero_division=0),
            'Macro Recall': recall_score(ydata, y_pred, average='macro', zero_division=0),
            'Macro F1': f1_score(ydata, y_pred, average='macro', zero_division=0)
        })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))

              Model Dataset  Accuracy  Macro Precision  Macro Recall  Macro F1
Logistic Regression   Train    0.8390           0.8385        0.8356    0.8367
Logistic Regression    Test    0.8374           0.8363        0.8345    0.8353
      Random Forest   Train    1.0000           1.0000        1.0000    1.0000
      Random Forest    Test    0.8844           0.8837        0.8824    0.8830
               GBDT   Train    0.8920           0.8923        0.8893    0.8905
               GBDT    Test    0.8870           0.8874        0.8839    0.8854


## Reflection

Which model would you choose based on the test results? What other checks would you want before using the model to support real lending decisions?

I would choose GBDT because it achieves the highest test accuracy (88.70%) and macro F1-score (0.8854). It also shows a relatively small gap between training and test performance, suggesting good generalisation. In contrast, Random Forest shows evidence of overfitting, with 100% training accuracy but only 88.44% test accuracy.

Before using GBDT for real lending decisions, I would examine the confusion matrix, particularly the costs of different classification errors. I would also assess fairness across demographic groups, model interpretability, and performance on new data. Further cross-validation would help confirm whether its small advantage over Random Forest is reliable.